# L07: Cluster the unlabelled data and defend the result

**Day 2.** Unsupervised learning with no ground truth: choosing k and committing to it before seeing profiles, K-Means and its local optima, other clustering families, and Isolation Forest anomaly detection.

- Reads: checkpoints/L02_clean.csv
- Writes: work/L07_clusters.csv, work/L07_anomalies.csv

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def k_scan(X, ks=range(2, 9), seed=2026, silhouette_sample=1500):
    """K-Means for each k: inertia (for the elbow) and silhouette (on a fixed random sample, for speed).

    n_init is set explicitly to 10 restarts per k. Lab L07.
    """
    rows = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
        sil = silhouette_score(X, km.labels_, sample_size=min(silhouette_sample, len(X)), random_state=seed)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": sil})
    return pd.DataFrame(rows).set_index("k")


def profile_clusters(df, labels, columns):
    """Size and median of each column, per cluster.

    Lab L07.
    """
    out = df[columns].groupby(np.asarray(labels)).median()
    out.insert(0, "size", pd.Series(np.asarray(labels)).value_counts().sort_index().to_numpy())
    out.index.name = "cluster"
    return out


def anomaly_rank(X, ids, n_estimators=500, seed=2026):
    """Isolation Forest anomaly scores, highest (most isolated) first.

    Returns a DataFrame of identifiers and scores, sorted. Lab L07.
    """
    forest = IsolationForest(n_estimators=n_estimators, random_state=seed).fit(X)
    scores = -forest.score_samples(X)
    return (pd.DataFrame({"id": np.asarray(ids), "anomaly_score": scores})
            .sort_values("anomaly_score", ascending=False).reset_index(drop=True))


NUMERIC = ["aircraft_age_years", "total_flight_hours", "days_since_inspection", "ground_time_hours",
           "cargo_weight_klb", "temperature_c", "sortie_duration_hours", "open_major", "open_minor"]

print("Helper functions ready")

## Part 1: Commit to k

**Specification.** Remove the outcome, standardise the numeric features, and scan k from 2 to 8. Commit to a number of
clusters, in writing, before any cluster profile is shown.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8")
# No outcome and no aircraft type: the algorithm sees only the nine numeric features.
X = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(df[NUMERIC]))
scan = k_scan(X, ks=range(2, 9), seed=SEED)
print(scan.round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.plot(scan.index, scan["inertia"], marker="o")
ax1.set_title("elbow: inertia")
ax2.plot(scan.index, scan["silhouette"], marker="o")
ax2.set_title("silhouette")
for ax in (ax1, ax2):
    ax.set_xlabel("k")
plt.tight_layout()
plt.show()

### Step 4 · DECIDE

In [ ]:
# DECIDE · DP-L07-1 · how many clusters, committed before any profile is shown.
# Replace ____ with your choice. The choices are taught on the slide "Choosing k: the elbow, the silhouette, and committing".
# Read the inertia and silhouette curves above, then commit. Any k you can defend in writing is accepted.
K_CHOSEN = ____
print("committed to k =", K_CHOSEN)

### Step 5 · WRITE

**WRITE · Written commitment**. State your k and your reason, before running the next cell. What would make you change your mind?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Fit and profile

**Specification.** Fit K-Means with the committed k, profile the clusters, and see how much the answer depends on the start.

### Step 6 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
km = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=SEED).fit(X)
profile = profile_clusters(df, km.labels_, NUMERIC)
print(profile.round(1).T)
print()
print("clusters against aircraft type, which the algorithm never saw:")
print(pd.crosstab(km.labels_, df["aircraft_type"]))

# K-Means only finds a local optimum: single starts from different seeds end in different places.
inertias = [KMeans(n_clusters=K_CHOSEN, n_init=1, random_state=s).fit(X).inertia_ for s in range(5)]
print("\ninertia from five single starts:", [round(v) for v in inertias], "| best of 10 starts:", round(km.inertia_))

### Step 7 · WRITE

**WRITE · your interpretation**. What do the clusters represent? Did you discover anything the group did not already know?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Other families

**Specification.** Compare K-Means with hierarchical clustering, a Gaussian mixture and DBSCAN on a random sample of 600.

### Step 8 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
sample = np.random.default_rng(SEED).choice(len(X), size=600, replace=False)
Xs, km_s = X[sample], km.labels_[sample]
others = {
    "hierarchical (Ward)": AgglomerativeClustering(n_clusters=K_CHOSEN, linkage="ward").fit_predict(Xs),
    "Gaussian mixture": GaussianMixture(n_components=K_CHOSEN, random_state=SEED).fit(Xs).predict(Xs),
    "DBSCAN (eps 2.0)": DBSCAN(eps=2.0, min_samples=10).fit_predict(Xs),
}
for name, labels in others.items():
    sizes = pd.Series(labels).value_counts().sort_index().to_dict()
    print(f"{name:<20} agreement with K-Means (adjusted Rand) {adjusted_rand_score(km_s, labels):.2f}; sizes {sizes}")

### Step 9 · WRITE

**WRITE · your interpretation**. The three methods disagree with K-Means to very different degrees. What does that tell you about the
cluster structure, and what does DBSCAN's label -1 mean?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: Anomaly detection

**Specification.** Rank every departure by how easily an Isolation Forest isolates it. Look at the top of the list.

### Step 10 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
ranked = anomaly_rank(X, df["mission_id"], n_estimators=500, seed=SEED)
top = ranked.head(10).merge(df, left_on="id", right_on="mission_id")
print(top[["id", "anomaly_score", "aircraft_type", "cargo_weight_klb", "ground_time_hours",
           "temperature_c", "days_since_inspection", "flag_out_of_range"]].round(3).to_string(index=False))

### Step 11 · WRITE

**WRITE · your interpretation**. Which departure ranks first, and how does it connect to Day 1? Why is an anomaly score a lead to
investigate rather than a verdict?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 12 · RUN

In [ ]:
# RUN · saves this part's output to work/.
pd.DataFrame({"mission_id": df["mission_id"], "cluster": km.labels_}).to_csv(
    WORK / "L07_clusters.csv", index=False, encoding="utf-8")
ranked.head(20).rename(columns={"id": "mission_id"}).round(5).to_csv(
    WORK / "L07_anomalies.csv", index=False, encoding="utf-8")
print("saved work/L07_clusters.csv and work/L07_anomalies.csv")